In [2]:
import numpy as np

# =====================================================================
# UPDATED TIMELINE PARAMETERS
# =====================================================================
T = 30.0       # Total time horizon = 30 min
nsteps = 30    # N = 30 discrete time steps (resulting in dt = 1.0 min)

# =====================================================================
# TARGET SETPOINTS (SP) FROM TABLE 2
# =====================================================================
SP = {
    'CV': [1.00 for _ in range(nsteps)],
    'Ln': [15.00 for _ in range(nsteps)]
}

# =====================================================================
# UPDATED ACTION SPACE (Incremental control bounds)
# =====================================================================
action_space = {
    'low': np.array([-1.0]),   # Minimum delta T step allowed per minute
    'high': np.array([1.0])    # Maximum delta T step allowed per minute
}

# =====================================================================
# OBSERVATION SPACE (State Bounds from Table 2 + Setpoints)
# =====================================================================
# Order: [mu0, mu1, mu2, mu3, C, CV, Ln, CV_SP, Ln_SP]
observation_space = {
    'low' : np.array([0.0, 0.0, 0.0, 0.0, 0.00, 0.00, 0.00, 0.00, 0.00]),
    'high' : np.array([1.0e20, 1.0e20, 1.0e20, 1.0e20, 0.50, 2.00, 20.00, 2.00, 20.00])  
}

# =====================================================================
# ENVIRONMENT PARAMETERS SETUP
# =====================================================================
env_params = {
    'N': nsteps, 
    'tsim': T, 
    'SP': SP, 
    'o_space': observation_space, 
    'a_space': action_space,
    
    # Initial conditions (x0) including the initial setpoints
    'x0': np.array([1.50e3, 2.30e4, 1.80e6, 2.50e8, 0.16, 1.00, 15.00, 1.00, 15.00]),
    
    'r_scale': {
        'CV': 1e1,
        'Ln': 1e0
    },
    
    'model': 'crystallization', 
    
    'normalise_a': True, 
    'normalise_o': True, 
    'noise': True, 
    'integration_method': 'jax', 
    'noise_percentage': 0.01, 
}

In [3]:
import pandas as pd
from pcgym import make_env
from tqdm import trange

# 1. Use your FULL 9-dimensional environment so you can capture the true mu3!
base_env = make_env(env_params)

# Storage lists for your dataset
dataset = []
num_episodes = 5000  # Run multiple batches to collect diverse data

for episode in trange(num_episodes):
    obs, info = base_env.reset()
    done = False
    truncated = False
    
    while not (done or truncated):
        # 2. Extract the physical layout of the current observation
        # obs order: [mu0, mu1, mu2, mu3, C, CV, Ln, CV_SP, Ln_SP]
        
        mu0, mu1, mu2, mu3, C, CV, Ln, CV_SP, Ln_SP = obs
        
        # 3. Save your measurable inputs (What the Soft Sensor will see)
        # You can choose to include mu0-mu2, or stick purely to easy measurements like C, CV, Ln.
        current_inputs = [mu0, mu1, mu2, C, CV, Ln, CV_SP, Ln_SP, mu3]
        
        dataset.append(current_inputs)
        
        # 5. Take an exploratory random action to gather diverse state space data
        random_action = base_env.action_space.sample() 
        obs, reward, done, truncated, info = base_env.step(random_action)

# 6. Convert to a Pandas DataFrame for easy Machine Learning training
feature_names = ['mu0', 'mu1', 'mu2', 'C', 'CV', 'Ln', 'CV_SP', 'Ln_SP', 'mu3']
dataset = pd.DataFrame(dataset, columns=feature_names)

print("Dataset ready!")
print("X Shape (Features):", dataset.shape)
print('-'*30)
print(dataset)
dataset.to_csv(f'cryst-{num_episodes}ep.csv', index=False)
print(f'Dataset is saved successfully with {num_episodes} episodes!!!')

/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
  0%|          | 0/5000 [00:00<?, ?it/s]

100%|██████████| 5000/5000 [1:16:05<00:00,  1.10it/s]  


Dataset ready!
X Shape (Features): (145000, 9)
------------------------------
        mu0  mu1  mu2         C        CV        Ln  CV_SP  Ln_SP  mu3
0      -1.0 -1.0 -1.0 -0.360000  0.000000  0.500000    0.0    0.5 -1.0
1      -1.0 -1.0 -1.0 -0.363845 -0.917765  1.725569    0.0    0.5 -1.0
2      -1.0 -1.0 -1.0 -0.362031 -1.154758  2.631652    0.0    0.5 -1.0
3      -1.0 -1.0 -1.0 -0.361980 -1.172934  2.943233    0.0    0.5 -1.0
4      -1.0 -1.0 -1.0 -0.382700 -1.095852  2.977309    0.0    0.5 -1.0
...     ...  ...  ...       ...       ...       ...    ...    ...  ...
144995 -1.0 -1.0 -1.0 -0.659736 -0.572898  0.549136    0.0    0.5 -1.0
144996 -1.0 -1.0 -1.0 -0.667695 -0.557340  0.515029    0.0    0.5 -1.0
144997 -1.0 -1.0 -1.0 -0.674181 -0.552504  0.505917    0.0    0.5 -1.0
144998 -1.0 -1.0 -1.0 -0.673231 -0.528849  0.471441    0.0    0.5 -1.0
144999 -1.0 -1.0 -1.0 -0.682444 -0.523362  0.440253    0.0    0.5 -1.0

[145000 rows x 9 columns]
Dataset is saved successfully with 5000 epi